# House Price Predictor: Model Training

This notebook generates a synthetic housing dataset, trains a Linear Regression model, evaluates its performance, and pushes the model artifacts to Hugging Face.

In [ ]:
# Install required dependencies in the Colab VM
!pip install pandas numpy scikit-learn huggingface_hub

## 1. Generate Synthetic Data
We generate 300 rows of synthetic house prices based on a linear combination of features plus Gaussian noise.

In [ ]:
import numpy as np
import pandas as pd
import os

# Seed for reproducibility
np.random.seed(42)
n_samples = 300

# Features
size_sqft = np.random.uniform(500, 4000, n_samples)
bedrooms = np.random.randint(1, 7, n_samples)
bathrooms = np.random.randint(1, 5, n_samples)
age_years = np.random.randint(0, 41, n_samples)
location_score = np.random.randint(1, 11, n_samples)

# Linear equation logic
base_price = 15.0
price_lakhs = (
    base_price +
    (size_sqft * 0.05) +
    (bedrooms * 5.0) +
    (bathrooms * 3.0) +
    (location_score * 4.0) -
    (age_years * 0.5)
)

# Gaussian noise
noise = np.random.normal(0, 12, n_samples)
price_lakhs = np.clip(price_lakhs + noise, 10.0, None)

# Construct DataFrame
df = pd.DataFrame({
    'size_sqft': np.round(size_sqft, 2),
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age_years': age_years,
    'location_score': location_score,
    'price_lakhs': np.round(price_lakhs, 2)
})

os.makedirs('data', exist_ok=True)
df.to_csv('data/house_prices.csv', index=False)
print("Dataset generated successfully and saved to data/house_prices.csv")
df.head()

## 2. Train the Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json

features = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years', 'location_score']
target = 'price_lakhs'

X = df[features]
y = df[target]

# Split data 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Model training complete.")
print(f"MAE:  {mae:.2f} Lakhs")
print(f"RMSE: {rmse:.2f} Lakhs")
print(f"R2:   {r2:.4f}")

# Save artifacts
os.makedirs('model', exist_ok=True)
with open('model/model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('model/feature_names.json', 'w') as f:
    json.dump(features, f)
print("Artifacts saved locally to model/ folder.")

## 3. Push Model to Hugging Face
Ensure you have logged in via CLI or run the login widget below.

In [ ]:
from huggingface_hub import HfApi
# If running manually in Colab, you can trigger Hugging Face login interactively:
# from huggingface_hub import notebook_login
# notebook_login()

api = HfApi()
username = api.whoami()['name']
repo_id = f"{username}/house-price-predictor"

print(f"Accessing repository: {repo_id}")
api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

print("Uploading model files...")
api.upload_folder(
    folder_path="model",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload trained Linear Regression model from Colab"
)
print(f"Model files pushed successfully! View at: https://huggingface.co/{repo_id}")